# Early identification of higher-education withdrawal risk

Logistic Regression and a small neural network are compared on two public datasets: **UCI** (dropout at enrolment) and **OULAD** (later withdrawal using only day-0–30 information). The neural network is the second algorithm so at least one model differs from typical Practical Data Science tree / kNN work. Thresholds are tuned on validation data; the test set is used once.

**Headline results.** Enrolment-time UCI models are stronger than day-30 OULAD models. After tuning, Logistic Regression is the better model on both sources (UCI recall 0.78, F1 0.72, PR-AUC 0.82; OULAD recall 0.60, F1 0.40, PR-AUC 0.34). The neural network is close but weaker (UCI PR-AUC 0.76; OULAD PR-AUC 0.32). The two datasets remain complementary: who looks risky at entry vs who disengages in the first 30 days.

The analysis is a proposed professional application relevant to the ACER Data Scientist role. It is not presented as an existing ACER project.

## 1. Setup and reproducibility

Resolve the project root whether the kernel starts in the repo or in `notebooks`. Keep `random_state=42` so the splits below match the reported metrics. Logistic Regression is the linear baseline; the small neural network is the second algorithm so at least one model differs from typical Practical Data Science tree / kNN work.

In [1]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, train_test_split

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.modeling as modeling
importlib.reload(modeling)

from src.data_prep import build_processed_tables  # ZIP files -> leakage-safe CSVs
from src.modeling import (
    evaluate_models,          # fit on train, score once on test
    extract_model_insights,   # LR coefficients or NN permutation importance
    make_models,              # LR + small neural-network pipelines
    metrics_at_threshold,     # accuracy / precision / recall / F1 / PR-AUC / ROC-AUC
    plot_threshold_curves,    # precision, recall, and F1 vs candidate cut-offs
    plot_top_features,        # bar chart of the strongest model contributions
    save_confusion_matrices,  # predicted vs actual heatmaps
    tune_threshold_for_f1,    # F1-maximising threshold on validation data only
)

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'figures'
RESULTS = ROOT / 'results'
for directory in [PROCESSED, FIGURES, RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 80)
ROOT

WindowsPath('C:/Users/ADM/Downloads/ACER_Student_Project')

## 2. Build or load the leakage-safe modelling tables

Processed tables: UCI 3,630 × 25 (enrolment features only) and OULAD 27,443 × 36 (background plus day-0–30 VLE activity). The first run builds them from the source ZIPs; later runs reload the CSVs.

In [2]:
uci_path = PROCESSED / 'uci_enrolment_model.csv'
oulad_path = PROCESSED / 'oulad_day30_model.csv'

if not (uci_path.exists() and oulad_path.exists()):
    uci_zip = RAW / 'predict+students+dropout+and+academic+success.zip'
    oulad_zip = RAW / 'open+university+learning+analytics+dataset.zip'
    missing = [str(path) for path in [uci_zip, oulad_zip] if not path.exists()]
    if missing:
        raise FileNotFoundError('Place the two source ZIP files in data/raw. Missing: ' + ', '.join(missing))
    build_processed_tables(uci_zip, oulad_zip, PROCESSED)

uci = pd.read_csv(uci_path)
oulad = pd.read_csv(oulad_path)
print(f'UCI shape: {uci.shape}')
print(f'OULAD shape: {oulad.shape}')

UCI shape: (3630, 25)
OULAD shape: (27443, 36)


## 3. Class balance

UCI dropouts are 39.1% of 3,630 students (1,421 cases). OULAD later withdrawals are 18.3% of 27,443 records (5,033 cases). Accuracy is a poor headline metric on OULAD, so the rest of the notebook reports recall, F1, and PR-AUC.

In [3]:
summary = pd.DataFrame({
    'dataset': ['UCI enrolment', 'OULAD day 30'],
    'observations': [len(uci), len(oulad)],
    'positive_cases': [uci['target_dropout'].sum(), oulad['target_withdrawn'].sum()],
    'positive_rate': [uci['target_dropout'].mean(), oulad['target_withdrawn'].mean()],
})
summary

,dataset,observations,positive_cases,positive_rate
0,UCI enrolment,3630,1421,0.391460
1,OULAD day 30,27443,5033,0.183398


In [4]:
assert set(uci['target_dropout'].unique()) <= {0, 1}
assert set(oulad['target_withdrawn'].unique()) <= {0, 1}
assert not any(col.startswith('curricular_units_') for col in uci.columns)
assert 'final_result' not in oulad.columns
assert 'date_unregistration' not in oulad.columns

plot_data = summary.assign(positive_rate_pct=summary['positive_rate'] * 100)
ax = sns.barplot(data=plot_data, x='dataset', y='positive_rate_pct', color='#1769aa')
ax.set(xlabel='', ylabel='Positive class (%)', title='Class balance after leakage-safe cohort construction')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')
plt.tight_layout()
plt.savefig(FIGURES / 'class_balance.png', dpi=180)
plt.show()

C:\Users\ADM\AppData\Local\Temp\ipykernel_21100\3842947970.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. UCI: enrolment-time modelling

Semester-result columns are excluded. A stratified 80/20 test split, then a validation slice of training, is used only to pick an F1-maximising threshold.

**Test results.** Default 0.5 is already near-optimal for both models (selected 0.475). Logistic Regression is the stronger UCI model: recall **0.78**, F1 0.72, PR-AUC 0.82. The neural network reaches recall 0.71, F1 0.67, PR-AUC 0.76. A linear enrolment-time signal is enough here; the extra nonlinearity does not improve ranking or recall.

In [5]:
uci_categorical = [
    'marital_status', 'application_mode', 'course',
    'daytime_evening_attendance', 'previous_qualification', 'nationality',
    'mother_s_qualification', 'father_s_qualification',
    'mother_s_occupation', 'father_s_occupation', 'displaced',
    'educational_special_needs', 'debtor', 'tuition_fees_up_to_date',
    'gender', 'scholarship_holder', 'international'
]
uci_numeric = [
    'application_order', 'previous_qualification_grade', 'admission_grade',
    'age_at_enrollment', 'unemployment_rate', 'inflation_rate', 'gdp'
]

x_uci = uci[uci_categorical + uci_numeric]
y_uci = uci['target_dropout']
x_uci_train, x_uci_test, y_uci_train, y_uci_test = train_test_split(
    x_uci, y_uci, test_size=0.20, stratify=y_uci, random_state=42
)
x_uci_fit, x_uci_val, y_uci_fit, y_uci_val = train_test_split(
    x_uci_train, y_uci_train, test_size=0.20, stratify=y_uci_train, random_state=42
)
print('Positive rates — fit / val / test:')
print(y_uci_fit.mean().round(4), y_uci_val.mean().round(4), y_uci_test.mean().round(4))

Positive rates — fit / val / test:
0.3917 0.3907 0.3912


In [6]:
# make_models: LR + a small neural net; LR is linear, the MLP can capture interactions.
uci_models_for_tuning = make_models(uci_categorical, uci_numeric)
uci_thresholds = {}

for name, model in uci_models_for_tuning.items():
    model.fit(x_uci_fit, y_uci_fit)
    val_proba = model.predict_proba(x_uci_val)[:, 1]
    # Scan validation probabilities and keep the cut-off that maximises F1.
    best_threshold, _scan = tune_threshold_for_f1(y_uci_val, val_proba)
    uci_thresholds[name] = best_threshold

# UCI threshold plots are omitted: the default 0.5 cut-off is already near-optimal.
pd.Series(uci_thresholds, name='selected_threshold').to_frame()

,selected_threshold
Logistic Regression,0.475
Neural Network,0.475


In [7]:
# evaluate_models fits on full training and scores the untouched test set once.
# First pass: default 0.5 cut-off. Second pass: validation-chosen thresholds.
uci_models = make_models(uci_categorical, uci_numeric)
uci_metrics_default, _, uci_matrices_default = evaluate_models(
    make_models(uci_categorical, uci_numeric),
    x_uci_train, y_uci_train, x_uci_test, y_uci_test,
)
uci_metrics_default.insert(0, 'dataset', 'UCI enrolment')
uci_metrics_default.insert(2, 'threshold_rule', 'default_0.5')

uci_metrics, uci_fitted, uci_matrices = evaluate_models(
    uci_models, x_uci_train, y_uci_train, x_uci_test, y_uci_test, thresholds=uci_thresholds
)
uci_metrics.insert(0, 'dataset', 'UCI enrolment')
uci_metrics.insert(2, 'threshold_rule', 'f1_max_on_validation')

uci_compare = pd.concat([uci_metrics_default, uci_metrics], ignore_index=True)
uci_compare.to_csv(RESULTS / 'uci_baseline_metrics.csv', index=False)


save_confusion_matrices('UCI', uci_matrices, FIGURES, suffix='tuned', show=True)
uci_compare.round(3)

C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:359: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:359: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,dataset,model,threshold_rule,threshold,accuracy,precision,recall,f1,pr_auc_average_precision,roc_auc
0,UCI enrolment,Logistic Regression,default_0.5,0.500,0.764,0.673,0.775,0.720,0.822,0.855
1,UCI enrolment,Neural Network,default_0.5,0.500,0.745,0.663,0.708,0.685,0.758,0.816
2,UCI enrolment,Logistic Regression,f1_max_on_validation,0.475,0.758,0.662,0.778,0.715,0.822,0.855
3,UCI enrolment,Neural Network,f1_max_on_validation,0.475,0.729,0.637,0.711,0.672,0.758,0.816


### 4.1 UCI model insights

Logistic Regression highlights specific courses, unpaid tuition, and application-mode/occupation codes. The neural network, via permutation importance, also ranks fee status, course, application mode, parental occupation, and scholarship holding. The two models agree on the same enrolment-time family of signals. These are predictive associations, not causes.

In [8]:
# extract_model_insights: signed LR coefficients, or NN permutation importance (PR-AUC drop).
uci_insight_frames = []
for name, model in uci_fitted.items():
    insights = extract_model_insights(model, top_n=12, x=x_uci_train, y=y_uci_train)
    insights.insert(0, 'dataset', 'UCI enrolment')
    insights.insert(1, 'model', name)
    uci_insight_frames.append(insights)

# One feature plot for the report: Neural Network (nonlinear UCI model).
plot_top_features(
    uci_insight_frames[1],
    title='UCI top contributions: Neural Network',
    output_path=FIGURES / 'uci_neural_network_top_features.png',
    show=True,
)

uci_insights = pd.concat(uci_insight_frames, ignore_index=True)
uci_insights.to_csv(RESULTS / 'uci_model_insights.csv', index=False)
uci_insights

C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:289: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,dataset,model,feature,value,abs_value,interpretation,source
0,UCI enrolment,Logistic Regression,course_9119,1.649027,1.649027,associated with higher risk,logistic_coefficient
1,UCI enrolment,Logistic Regression,tuition_fees_up_to_date_0,1.549649,1.549649,associated with higher risk,logistic_coefficient
2,UCI enrolment,Logistic Regression,mother_s_occupation_0,1.465159,1.465159,associated with higher risk,logistic_coefficient
3,UCI enrolment,Logistic Regression,tuition_fees_up_to_date_1,-1.460308,1.460308,associated with lower risk,logistic_coefficient
4,UCI enrolment,Logistic Regression,mother_s_occupation_191,-1.397926,1.397926,associated with lower risk,logistic_coefficient
5,UCI enrolment,Logistic Regression,application_mode_7,1.217668,1.217668,associated with higher risk,logistic_coefficient
6,UCI enrolment,Logistic Regression,course_9853,1.154911,1.154911,associated with higher risk,logistic_coefficient
7,UCI enrolment,Logistic Regression,application_mode_15,-1.083546,1.083546,associated with lower risk,logistic_coefficient
8,UCI enrolment,Logistic Regression,nationality_26,-1.038199,1.038199,associated with lower risk,logistic_coefficient
9,UCI enrolment,Logistic Regression,application_mode_39,1.033509,1.033509,associated with higher risk,logistic_coefficient


## 5. OULAD: day-30 modelling

Features are background plus VLE activity through day 30. Students already withdrawn by day 30 are excluded. Splits are grouped by `id_student` (train 21,936 / test 5,507; fit 17,528 / val 4,408), with ~18% positive rate in each fold.

**Test results.** Logistic Regression is usable at 0.5 (recall 0.64) and remains best after tuning (threshold 0.525; F1 **0.40**; PR-AUC 0.34; recall 0.60). The neural network stays at threshold 0.5 (recall 0.58, F1 0.37, PR-AUC 0.32). Precision is low for both (~0.27–0.30), so many flagged students would not withdraw.

In [9]:
oulad_categorical = [
    'code_module', 'code_presentation', 'gender', 'region',
    'highest_education', 'imd_band', 'age_band', 'disability'
]
oulad_excluded = {'id_student', 'target_withdrawn', *oulad_categorical}
oulad_numeric = [col for col in oulad.columns if col not in oulad_excluded]

x_oulad = oulad[oulad_categorical + oulad_numeric]
y_oulad = oulad['target_withdrawn']
groups = oulad['id_student']

group_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(group_split.split(x_oulad, y_oulad, groups=groups))
x_oulad_train, x_oulad_test = x_oulad.iloc[train_idx], x_oulad.iloc[test_idx]
y_oulad_train, y_oulad_test = y_oulad.iloc[train_idx], y_oulad.iloc[test_idx]
groups_train = groups.iloc[train_idx]

val_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
fit_idx, val_idx = next(val_split.split(x_oulad_train, y_oulad_train, groups=groups_train))
x_oulad_fit, x_oulad_val = x_oulad_train.iloc[fit_idx], x_oulad_train.iloc[val_idx]
y_oulad_fit, y_oulad_val = y_oulad_train.iloc[fit_idx], y_oulad_train.iloc[val_idx]

assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
assert set(groups_train.iloc[fit_idx]).isdisjoint(set(groups_train.iloc[val_idx]))
print(f'Train records: {len(train_idx):,}; test records: {len(test_idx):,}')
print(f'Fit / val records: {len(fit_idx):,} / {len(val_idx):,}')
print(
    'Positive rates — fit / val / test: '
    f'{y_oulad_fit.mean():.2%} / {y_oulad_val.mean():.2%} / {y_oulad_test.mean():.2%}'
)

Train records: 21,936; test records: 5,507
Fit / val records: 17,528 / 4,408
Positive rates — fit / val / test: 18.39% / 18.01% / 18.43%


In [10]:
# Same protocol as UCI: fit on the fit subset, choose the F1-max threshold on validation.
oulad_models_for_tuning = make_models(oulad_categorical, oulad_numeric)
oulad_thresholds = {}
oulad_threshold_scans = {}

for name, model in oulad_models_for_tuning.items():
    model.fit(x_oulad_fit, y_oulad_fit)
    val_proba = model.predict_proba(x_oulad_val)[:, 1]
    best_threshold, scan = tune_threshold_for_f1(y_oulad_val, val_proba)
    oulad_thresholds[name] = best_threshold
    oulad_threshold_scans[name] = scan


# Keep the Neural Network scan so the 0.5 vs tuned operating point is visible.
plot_threshold_curves(
    oulad_threshold_scans['Neural Network'],
    title='OULAD validation threshold scan: Neural Network',
    output_path=FIGURES / 'oulad_neural_network_threshold_scan.png',
    selected_threshold=oulad_thresholds['Neural Network'],
    show=True,
)

pd.Series(oulad_thresholds, name='selected_threshold').to_frame()

C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:322: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,selected_threshold
Logistic Regression,0.525
Neural Network,0.500


In [11]:
# Default 0.5 vs validation-tuned thresholds on the untouched test set.
oulad_metrics_default, _, oulad_matrices_default = evaluate_models(
    make_models(oulad_categorical, oulad_numeric),
    x_oulad_train, y_oulad_train, x_oulad_test, y_oulad_test,
)
oulad_metrics_default.insert(0, 'dataset', 'OULAD day 30')
oulad_metrics_default.insert(2, 'threshold_rule', 'default_0.5')

oulad_models = make_models(oulad_categorical, oulad_numeric)
oulad_metrics, oulad_fitted, oulad_matrices = evaluate_models(
    oulad_models,
    x_oulad_train,
    y_oulad_train,
    x_oulad_test,
    y_oulad_test,
    thresholds=oulad_thresholds,
)
oulad_metrics.insert(0, 'dataset', 'OULAD day 30')
oulad_metrics.insert(2, 'threshold_rule', 'f1_max_on_validation')

oulad_compare = pd.concat([oulad_metrics_default, oulad_metrics], ignore_index=True)
oulad_compare.to_csv(RESULTS / 'oulad_baseline_metrics.csv', index=False)

# Default-0.5 confusion for the Neural Network (before/after threshold); keep all tuned matrices.
save_confusion_matrices(
    'OULAD',
    {'Neural Network': oulad_matrices_default['Neural Network']},
    FIGURES,
    suffix='default_0.5',
    show=True,
)
save_confusion_matrices('OULAD', oulad_matrices, FIGURES, suffix='tuned', show=True)
oulad_compare.round(3)

C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:359: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:359: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:359: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,dataset,model,threshold_rule,threshold,accuracy,precision,recall,f1,pr_auc_average_precision,roc_auc
0,OULAD day 30,Logistic Regression,default_0.5,0.500,0.639,0.286,0.637,0.395,0.343,0.689
1,OULAD day 30,Neural Network,default_0.5,0.500,0.640,0.274,0.579,0.372,0.321,0.667
2,OULAD day 30,Logistic Regression,f1_max_on_validation,0.525,0.666,0.297,0.595,0.396,0.343,0.689
3,OULAD day 30,Neural Network,f1_max_on_validation,0.500,0.640,0.274,0.579,0.372,0.321,0.667


### 5.1 OULAD model insights

Logistic Regression treats unique sites and active days as protective, but module and education codes dominate its coefficients (GGG lower risk; CCC/DDD higher). The neural network also uses module, education, and active days, and additionally ranks region, IMD band, and disability highly. Demographic fields would need fairness review before operational use.

In [12]:
# Same insight extractors as UCI: LR direction, NN permutation importance.
oulad_insight_frames = []
for name, model in oulad_fitted.items():
    insights = extract_model_insights(model, top_n=12, x=x_oulad_train, y=y_oulad_train)
    insights.insert(0, 'dataset', 'OULAD day 30')
    insights.insert(1, 'model', name)
    oulad_insight_frames.append(insights)

# One feature plot for the report: Neural Network highlights module, activity, and demographic signals.
plot_top_features(
    oulad_insight_frames[1],
    title='OULAD top contributions: Neural Network',
    output_path=FIGURES / 'oulad_neural_network_top_features.png',
    show=True,
)

oulad_insights = pd.concat(oulad_insight_frames, ignore_index=True)
oulad_insights.to_csv(RESULTS / 'oulad_model_insights.csv', index=False)
oulad_insights

C:\Users\ADM\Downloads\ACER_Student_Project\src\modeling.py:289: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,dataset,model,feature,value,abs_value,interpretation,source
0,OULAD day 30,Logistic Regression,code_module_GGG,-1.123737,1.123737,associated with lower risk,logistic_coefficient
1,OULAD day 30,Logistic Regression,code_module_CCC,0.891112,0.891112,associated with higher risk,logistic_coefficient
2,OULAD day 30,Logistic Regression,code_module_BBB,-0.562494,0.562494,associated with lower risk,logistic_coefficient
3,OULAD day 30,Logistic Regression,code_module_DDD,0.559344,0.559344,associated with higher risk,logistic_coefficient
4,OULAD day 30,Logistic Regression,highest_education_No Formal quals,0.381257,0.381257,associated with higher risk,logistic_coefficient
5,OULAD day 30,Logistic Regression,code_module_FFF,0.322277,0.322277,associated with higher risk,logistic_coefficient
6,OULAD day 30,Logistic Regression,unique_sites,-0.319722,0.319722,associated with lower risk,logistic_coefficient
7,OULAD day 30,Logistic Regression,active_days,-0.275082,0.275082,associated with lower risk,logistic_coefficient
8,OULAD day 30,Logistic Regression,disability_Y,0.259033,0.259033,associated with higher risk,logistic_coefficient
9,OULAD day 30,Logistic Regression,highest_education_Post Graduate Qualification,-0.258672,0.258672,associated with lower risk,logistic_coefficient


## 6. Compare evidence across datasets

The datasets are not interchangeable: different countries, study modes, prevalence (39% vs 18%), and prediction timing.

**What holds together.** Both sources yield a usable risk ranking, but UCI enrolment-time prediction is much stronger (PR-AUC 0.82 vs 0.34). Logistic Regression outperforms the small neural network on every headline metric in both settings. The settings are complementary: who looks risky at entry vs who disengages in the first 30 days.

**Where they diverge.** The preferred algorithm does not change across datasets (Logistic Regression wins both), but the feature stories do: UCI is dominated by fee status and course/application codes; OULAD mixes module, early activity, and demographic fields. A small neural network does not overtake the linear baseline on these tables, which is a limitation of the second model rather than a reason to drop the enrolment-time vs day-30 design.

In [13]:
all_metrics = pd.concat([uci_compare, oulad_compare], ignore_index=True)
all_metrics.to_csv(RESULTS / 'all_baseline_metrics.csv', index=False)

all_insights = pd.concat([uci_insights, oulad_insights], ignore_index=True)
all_insights.to_csv(RESULTS / 'all_model_insights.csv', index=False)

tuned = all_metrics.loc[all_metrics['threshold_rule'] == 'f1_max_on_validation'].copy()
default = all_metrics.loc[all_metrics['threshold_rule'] == 'default_0.5'].copy()

comparison_plot = tuned.melt(
    id_vars=['dataset', 'model'],
    value_vars=['recall', 'precision', 'f1', 'pr_auc_average_precision'],
    var_name='metric',
    value_name='score',
)
g = sns.catplot(
    data=comparison_plot,
    x='metric',
    y='score',
    hue='model',
    col='dataset',
    kind='bar',
    height=4.2,
    aspect=1.15,
)
g.set_titles('{col_name}')
g.set_axis_labels('', 'Score')
g.set(ylim=(0, 1))
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=25)
g.fig.suptitle('Tuned-threshold test metrics by dataset', y=1.03)
g.savefig(FIGURES / 'tuned_metrics_comparison.png', dpi=180)
plt.show()

all_metrics.round(3)

C:\Users\ADM\AppData\Local\Temp\ipykernel_21100\2719747483.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,dataset,model,threshold_rule,threshold,accuracy,precision,recall,f1,pr_auc_average_precision,roc_auc
0,UCI enrolment,Logistic Regression,default_0.5,0.500,0.764,0.673,0.775,0.720,0.822,0.855
1,UCI enrolment,Neural Network,default_0.5,0.500,0.745,0.663,0.708,0.685,0.758,0.816
2,UCI enrolment,Logistic Regression,f1_max_on_validation,0.475,0.758,0.662,0.778,0.715,0.822,0.855
3,UCI enrolment,Neural Network,f1_max_on_validation,0.475,0.729,0.637,0.711,0.672,0.758,0.816
4,OULAD day 30,Logistic Regression,default_0.5,0.500,0.639,0.286,0.637,0.395,0.343,0.689
5,OULAD day 30,Neural Network,default_0.5,0.500,0.640,0.274,0.579,0.372,0.321,0.667
6,OULAD day 30,Logistic Regression,f1_max_on_validation,0.525,0.666,0.297,0.595,0.396,0.343,0.689
7,OULAD day 30,Neural Network,f1_max_on_validation,0.500,0.640,0.274,0.579,0.372,0.321,0.667


In [14]:
# One row per dataset: which tuned model wins on recall / F1 / PR-AUC,
# and how Neural Network recall moves after threshold tuning.
findings = []

for dataset_name, default_frame, tuned_frame in [
    ('UCI enrolment', uci_metrics_default, uci_metrics),
    ('OULAD day 30', oulad_metrics_default, oulad_metrics),
]:
    best_recall_model = tuned_frame.loc[tuned_frame['recall'].idxmax(), 'model']
    best_f1_model = tuned_frame.loc[tuned_frame['f1'].idxmax(), 'model']
    best_pr_model = tuned_frame.loc[tuned_frame['pr_auc_average_precision'].idxmax(), 'model']

    nn_default = default_frame.loc[default_frame['model'] == 'Neural Network'].iloc[0]
    nn_tuned = tuned_frame.loc[tuned_frame['model'] == 'Neural Network'].iloc[0]
    lr_tuned = tuned_frame.loc[tuned_frame['model'] == 'Logistic Regression'].iloc[0]

    findings.append({
        'dataset': dataset_name,
        'best_recall_model_tuned': best_recall_model,
        'best_f1_model_tuned': best_f1_model,
        'best_pr_auc_model': best_pr_model,
        'nn_recall_default_0.5': nn_default['recall'],
        'nn_recall_tuned': nn_tuned['recall'],
        'lr_recall_tuned': lr_tuned['recall'],
        'lr_precision_tuned': lr_tuned['precision'],
        'nn_precision_tuned': nn_tuned['precision'],
    })

findings_frame = pd.DataFrame(findings)
findings_frame.to_csv(RESULTS / 'cross_dataset_findings.csv', index=False)
findings_frame.round(3)

,dataset,best_recall_model_tuned,best_f1_model_tuned,best_pr_auc_model,nn_recall_default_0.5,nn_recall_tuned,lr_recall_tuned,lr_precision_tuned,nn_precision_tuned
0,UCI enrolment,Logistic Regression,Logistic Regression,Logistic Regression,0.708,0.711,0.778,0.662,0.637
1,OULAD day 30,Logistic Regression,Logistic Regression,Logistic Regression,0.579,0.579,0.595,0.297,0.274
